In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import random

# Нейронная сеть
neural_network = nn.Sequential(
    nn.Linear(18, 64),    # Полносвязный слой с 18 входами и 64 выходами
    nn.ReLU(),
    nn.Linear(64, 32),    # Полносвязный слой с 64 входами и 32 выходами
    nn.ReLU(),
    nn.Linear(32, 6),     # Полносвязный слой с 32 входами и 6 выходами (по одному на каждый столбец)
    nn.Softmax(dim=-1)    # Функция активации Softmax для получения вероятностей действий
)

# Создание пустого игрового поля
def create_game_board():
    return np.zeros((3, 6), dtype=int)  # Возвращает пустое поле 3x6, заполненное нулями

# Совершение хода в указанный столбец
def make_move(board, col, player):
    for row in reversed(range(3)):    # Проходим по строкам снизу вверх
        if board[row, col] == 0:
            board[row, col] = player
            return True
    return False

# Проверка доступности хода в указанном столбце
def is_column_available(board, col):
    return board[0, col] == 0

# Получение списка всех доступных ходов
def get_available_moves(board):
    return [col for col in range(6) if is_column_available(board, col)]  # Возвращает список доступных столбцов для хода

# Проверка на ничью (все ячейки заполнены)
def is_board_full(board):
    return np.all(board != 0)

# Ход болвана
def bolvan(board):
    available_moves = get_available_moves(board)
    if available_moves:
        move = random.choice(available_moves)
        make_move(board, move, -1)

# Функция подсчёта комбинаций из трёх фишек подряд
def count_three_in_row(board, player):
    board = np.array(board).reshape(3, 6)
    count = 0
    # Проверка горизонтальных комбинаций
    for row in board:
        for i in range(4):
            if all(row[i:i+3] == player):
                count += 1
    # Проверка вертикальных комбинаций
    for col in range(6):
        for i in range(1):
            if all(board[i:i+3, col] == player):
                count += 1

    # Проверка диагональных комбинаций (слева направо и справа налево)
    for i in range(1):
        for j in range(4):
            if all([board[i+k, j+k] == player for k in range(3)]):
                count += 1
            if all([board[i+2-k, j+k] == player for k in range(3)]):
                count += 1
    return count

# Функция вывода текущего состояния поля в консоль
def display_board(board):
    for row in board:
        # Отображение доски
        formatted_row = " ".join(f"{cell:>3}" for cell in row)
        print(formatted_row)
    print()

In [ ]:
optimizer = optim.Adam(neural_network.parameters(), lr=0.0001)

epochs = 250000
invalid_move_count = 0

# Обучение с целью минимизации появления недопустимых ходов
for epoch in range(epochs):
    board = create_game_board()
    traectory = []                                  # Список для хранения траектории (состояние, действие, награда)
    current_player = random.choice([1, -1])
    game_over = False

    while not game_over:
        if current_player == 1:
            state = torch.tensor(board.flatten(), dtype=torch.float32)          # Преобразуем поле в тензор
            action_probs = neural_network(state)                                # Получаем вероятности действий от модели
            action_distribution = Categorical(action_probs)                     # Создаем категориальное распределение
            chosen_action = action_distribution.sample()                        # Выбираем действие на основе вероятностей

            if is_column_available(board, chosen_action.item()):
                make_move(board, chosen_action.item(), current_player)
                reward = 1
                traectory.append((state, chosen_action, reward))
                current_player = -1
            else:
                reward = -100
                invalid_move_count += 1
                traectory.append((state, chosen_action, reward))
                if not get_available_moves(board):
                    game_over = True
                continue                                                        # Пропускаем оставшуюся часть цикла
        else:
            bolvan(board)
            current_player = 1

        if is_board_full(board):
            game_over = True

    if traectory:  # Если траектория не пуста

        states, actions, rewards = zip(*traectory)  # Разделяем траекторию на состояния, действия и награды
        # Объединяем состояния и действия в разные тензоры, а награды в массив NumPy
        states = torch.stack(states)
        actions = torch.stack(actions)
        rewards = np.array(rewards)

        # Вычисление наград НЕ ДИСКОНТИРОВАННЫХ!
        discounted_rewards = np.zeros_like(rewards, dtype=np.float32)
        cumulative_reward = 0
        for i in reversed(range(len(rewards))):
            cumulative_reward = rewards[i] + cumulative_reward
            discounted_rewards[i] = cumulative_reward

        # Нормализация наград
        discounted_rewards = (discounted_rewards - discounted_rewards.mean()) / (discounted_rewards.std() + 1e-9)
        discounted_rewards = torch.tensor(discounted_rewards)

        # Вычисление потерь
        action_probs = neural_network(states)                                   # Получаем вероятности действий для всех состояний
        action_distribution = Categorical(action_probs)                         # Создаем категориальное распределение
        log_probs = -action_distribution.log_prob(actions)                      # Вычисляем логарифмы вероятностей выбранных действий
        loss = torch.sum(log_probs * discounted_rewards)  # Вычисляем потери

        # Обновление модели
        optimizer.zero_grad()   # Обнуляем градиенты
        loss.backward()         # Вычисляем градиенты
        optimizer.step()        # Обновляем параметры модели

    if (epoch + 1) % 1000 == 0:                                             # Каждые 1000 эпизодов
        print(f"Эпизод {epoch + 1}: Недопустимых ходов = {invalid_move_count}")  # Выводим статистику
        invalid_move_count = 0                                                     # Сбрасываем счетчик недопустимых ходов

    if (epoch + 1) % 10000 == 0:                            # Каждые 10000 эпизодов
        torch.save(neural_network, f"skorobogatov_david_{epoch + 1}.pt")  # Сохраняем модель
        print(f"Модель сохранена на эпизоде {epoch + 1}")   # Выводим сообщение о сохранении

Эпизод 1000: Недопустимых ходов = 7438
Эпизод 2000: Недопустимых ходов = 7742
Эпизод 3000: Недопустимых ходов = 7302
Эпизод 4000: Недопустимых ходов = 7569
Эпизод 5000: Недопустимых ходов = 7382
Эпизод 6000: Недопустимых ходов = 7268
Эпизод 7000: Недопустимых ходов = 7399
Эпизод 8000: Недопустимых ходов = 7456
Эпизод 9000: Недопустимых ходов = 7070
Эпизод 10000: Недопустимых ходов = 7368
Модель сохранена на эпизоде 10000
Эпизод 11000: Недопустимых ходов = 7257
Эпизод 12000: Недопустимых ходов = 7062
Эпизод 13000: Недопустимых ходов = 7106
Эпизод 14000: Недопустимых ходов = 7460
Эпизод 15000: Недопустимых ходов = 7370
Эпизод 16000: Недопустимых ходов = 7383
Эпизод 17000: Недопустимых ходов = 7327
Эпизод 18000: Недопустимых ходов = 7507
Эпизод 19000: Недопустимых ходов = 7463
Эпизод 20000: Недопустимых ходов = 6875
Модель сохранена на эпизоде 20000
Эпизод 21000: Недопустимых ходов = 7294
Эпизод 22000: Недопустимых ходов = 7072
Эпизод 23000: Недопустимых ходов = 7513
Эпизод 24000: Недопус

In [ ]:
# Функция для тестирования модели
def test_neural_network(model_path, num_games=10000):
    # Загрузка модели
    neural_network = torch.load(model_path, map_location=torch.device('cpu'), weights_only=False)  # Загружаем модель из файла
    neural_network.eval()  # Переводим модель в режим оценки

    # Статистика
    wins, losses, draws, incorrect_moves = 0, 0, 0, 0  # Инициализация счетчиков для побед, поражений, ничьих и неправильных ходов

    # Основной цикл игры
    for game in range(num_games):
        board = create_game_board()
        current_player = random.choice([1, -1])
        game_over = False

        while not game_over:
            if current_player == 1:  # Ход модели
                state = torch.tensor(board.flatten(), dtype=torch.float32)   # Преобразуем поле в тензор
                action_probs = neural_network(state)                                        # Получаем вероятности действий от модели
                chosen_action = torch.argmax(action_probs).item()                         # Выбираем ход с максимальной вероятностью

                if is_column_available(board, chosen_action):
                    make_move(board, chosen_action, current_player)
                else:
                    incorrect_moves += 1
                    break                                                       # Завершаем игру, если ход недопустим
                current_player = -1
            else:
                bolvan(board)
                current_player = 1

            # Проверка на заполнение поля
            if is_board_full(board):  # Если поле заполнено
                game_over = True         # Завершаем игру
                # Подсчёт комбинаций
                model_combinations = count_three_in_row(board, 1)    # Количество комбинаций модели
                opponent_combinations = count_three_in_row(board, -1)  # Количество комбинаций случайного игрока

                # Определение результата игры
                if model_combinations > opponent_combinations:
                    wins += 1
                elif opponent_combinations > model_combinations:
                    losses += 1
                else:
                    draws += 1

    # Вывод статистики
    print(f"Игр: {num_games}\nПобед: {wins}\nПоражений: {losses}\nНичьи: {draws}\nНеправильные ходы: {incorrect_moves}")

# Тест
# Вызов функции проверки
print("Тесты на 1000 игр\n-------------------------")
for i in range (10000,250001,10000):
    print(f"Test for skorobogatov_david_{i}.pt")
    test_neural_network(f"skorobogatov_david_{i}.pt", num_games=1000)
    print("-------------------------")

Тесты на 1000 игр
-------------------------
Test for skorobogatov_david_10000.pt
Игр: 1000
Побед: 1
Поражений: 1
Ничьи: 2
Неправильные ходы: 996
-------------------------
Test for skorobogatov_david_20000.pt
Игр: 1000
Побед: 4
Поражений: 5
Ничьи: 6
Неправильные ходы: 985
-------------------------
Test for skorobogatov_david_30000.pt
Игр: 1000
Побед: 4
Поражений: 10
Ничьи: 8
Неправильные ходы: 978
-------------------------
Test for skorobogatov_david_40000.pt
Игр: 1000
Побед: 18
Поражений: 19
Ничьи: 19
Неправильные ходы: 944
-------------------------
Test for skorobogatov_david_50000.pt
Игр: 1000
Побед: 80
Поражений: 247
Ничьи: 107
Неправильные ходы: 566
-------------------------
Test for skorobogatov_david_60000.pt
Игр: 1000
Побед: 71
Поражений: 594
Ничьи: 88
Неправильные ходы: 247
-------------------------
Test for skorobogatov_david_70000.pt
Игр: 1000
Побед: 64
Поражений: 695
Ничьи: 91
Неправильные ходы: 150
-------------------------
Test for skorobogatov_david_80000.pt
Игр: 1000
Поб

In [ ]:
# Обучение модели выигрышной стратегии
# Загрузка модели
neural_network = torch.load("skorobogatov_david_250000.pt", map_location=torch.device('cpu'), weights_only=False)  # Загружаем модель из файла
neural_network.train()  # Переводим модель в режим обучения

# Оптимизатор Adam с learning rate 0.0001
optimizer = optim.Adam(neural_network.parameters(), lr=0.0001)

# Параметры обучения
#discount_factor = 0.99       # Коэффициент дисконтирования для наград
learning_rate = 0.9
epochs = 100000

# Счетчики для статистики
wins, losses, draws, incorrect_moves = 0, 0, 0, 0

# Дообучение модели
for epoch in range(epochs):
    board = create_game_board()
    traectory = []                     # Список для хранения траектории (состояние, действие, награда)
    current_player = 1

    while True:  # Цикл игры
        if current_player == -1:
            bolvan(board)
        else:
            state = torch.tensor(board.flatten(), dtype=torch.float32)  # Преобразуем поле в тензор
            action_probs = neural_network(state)             # Получаем вероятности действий от модели
            action_distribution = Categorical(action_probs)  # Создаем категориальное распределение
            chosen_action = action_distribution.sample()     # Выбираем действие на основе вероятностей

            if not is_column_available(board, chosen_action.item()):
                reward = -100
                traectory.append((state.clone(), chosen_action, reward))
                incorrect_moves += 1
                break  # Завершаем игру
            make_move(board, chosen_action.item(), current_player)
            reward = 0
            traectory.append((state.clone(), chosen_action, reward))

        # Проверка на заполнение доски
        if is_board_full(board):
            # Подсчет комбинаций для обоих игроков
            model_combinations = count_three_in_row(board, 1)
            opponent_combinations = count_three_in_row(board, -1)

            # Определение награды
            if model_combinations > opponent_combinations:
                reward = 2 * (model_combinations - opponent_combinations)  # Награда за победу
                wins += 1
            elif model_combinations < opponent_combinations:
                reward = -2 * (opponent_combinations - model_combinations)  # Штраф за поражение
                losses += 1
            else:
                reward = -1
                draws += 1

            # Обновление наград в траектории
            for i in range(len(traectory)):  # Проходим по всей траектории
                traectory[i] = (traectory[i][0], traectory[i][1], reward)  # Обновляем награды
            break  # Завершаем игру

        current_player *= -1  # Смена игрока

    # Обновление модели
    loss = 0.0
    trajectory_length = len(traectory)
    for t in range(trajectory_length):

        #discounted_reward = 0.0  # Инициализация дисконтированной награды
        #for i in range(t, trajectory_length):  # Суммируем награды с учетом дисконтирования
            #discounted_reward += (discount_factor ** (i - t)) * traectory[i][2]  # Дисконтированная награда
        discounted_reward = sum(traectory[i][2] for i in range(t, trajectory_length)) # Не дисконтированная награда
        loss += -learning_rate * discounted_reward * Categorical(neural_network(traectory[t][0])).log_prob(traectory[t][1])  # Вычисление потерь

    optimizer.zero_grad()  # Обнуляем градиенты
    loss.backward()  # Вычисляем градиенты
    optimizer.step()  # Обновляем параметры модели

    # Вывод статистики и доски каждые 1000 эпох
    if (epoch + 1) % 1000 == 0:  # Каждые 1000 эпох
        print(f"Эпизод {epoch + 1}\nПобеды: {wins}, Поражения: {losses}, Ничьи: {draws}, Неправильные ходы: {incorrect_moves}\nФинальное поле:")
        wins, losses, draws, incorrect_moves = 0, 0, 0, 0  # Сбрасываем счетчики побед, поражений, ничьих и неправильных ходов
        display_board(board)  # Выводим финальное поле
        print(f"Кол-во комбинаций:\n Нейронная сеть - {model_combinations}\n Случайный игрок - {opponent_combinations}")
        print(20 * "_")

    if (epoch + 1) % 10000 == 0:  # Каждые 10000 эпох
        # Сохранение дообученной модели
        torch.save(neural_network, f"skorobogatov_david_final_{epoch + 1}.pt")  # Сохраняем модель

Эпизод 1000
Победы: 225, Поражения: 348, Ничьи: 159, Неправильные ходы: 268
Финальное поле:
  1  -1  -1  -1  -1   1
  1   1  -1  -1   1   1
  1  -1   1   1  -1  -1

Кол-во комбинаций:
 Нейронная сеть - 3
 Случайный игрок - 4
____________________
Эпизод 2000
Победы: 258, Поражения: 361, Ничьи: 150, Неправильные ходы: 231
Финальное поле:
 -1   1  -1   1   1  -1
  1  -1   1  -1   1   1
  1  -1   1  -1  -1  -1

Кол-во комбинаций:
 Нейронная сеть - 0
 Случайный игрок - 2
____________________
Эпизод 3000
Победы: 343, Поражения: 276, Ничьи: 154, Неправильные ходы: 227
Финальное поле:
  1   1  -1  -1  -1  -1
  1  -1   1  -1  -1   1
  1  -1   1   1   1  -1

Кол-во комбинаций:
 Нейронная сеть - 3
 Случайный игрок - 3
____________________
Эпизод 4000
Победы: 382, Поражения: 245, Ничьи: 168, Неправильные ходы: 205
Финальное поле:
 -1  -1   1   1  -1   1
 -1   1   1  -1   1  -1
  1  -1  -1   1  -1   1

Кол-во комбинаций:
 Нейронная сеть - 3
 Случайный игрок - 1
____________________
Эпизод 5000
Побе

In [ ]:
# Вызов функции проверки
print("Тесты на 1000 игр\n-------------------------")
for i in range (10000,100001,10000):
    print(f"Тест файла skorobogatov_david_final_{i}.pt")
    test_neural_network(f"skorobogatov_david_final_{i}.pt", num_games=1000)
    print("-------------------------")

Тесты на 1000 игр
-------------------------
Тест файла skorobogatov_david_final_10000.pt
Игр: 1000
Побед: 768
Поражений: 106
Ничьи: 126
Неправильные ходы: 0
-------------------------
Тест файла skorobogatov_david_final_20000.pt
Игр: 1000
Побед: 767
Поражений: 100
Ничьи: 133
Неправильные ходы: 0
-------------------------
Тест файла skorobogatov_david_final_30000.pt
Игр: 1000
Побед: 833
Поражений: 54
Ничьи: 113
Неправильные ходы: 0
-------------------------
Тест файла skorobogatov_david_final_40000.pt
Игр: 1000
Побед: 871
Поражений: 43
Ничьи: 86
Неправильные ходы: 0
-------------------------
Тест файла skorobogatov_david_final_50000.pt
Игр: 1000
Побед: 883
Поражений: 31
Ничьи: 86
Неправильные ходы: 0
-------------------------
Тест файла skorobogatov_david_final_60000.pt
Игр: 1000
Побед: 877
Поражений: 42
Ничьи: 81
Неправильные ходы: 0
-------------------------
Тест файла skorobogatov_david_final_70000.pt
Игр: 1000
Побед: 907
Поражений: 27
Ничьи: 66
Неправильные ходы: 0
------------------